In [ ]:
import librosa
import numpy as np
import os
import pandas as pd
import glob

input_dir = "extracted_audios"
pattern = os.path.join(input_dir, "**", "*.wav")
wav_files = glob.glob(pattern, recursive=True)
print(f"Found {len(wav_files)} .wav files")

data = []

for path in wav_files:
    try:
        y, sr = librosa.load(path, sr=None)
        if y.size == 0:
            print(f"Empty file: {path}")
            continue
        
        # Compute Mel spectrogram (default: 128 Mel bands)
        mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128, fmax=sr//2)
        
        # Convert power spectrogram (amplitude squared) to dB scale (log scale)
        mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
        
        # Summarize the Mel spectrogram by taking mean and std over time axis (axis=1)
        mel_mean = mel_spec_db.mean(axis=1)
        mel_std = mel_spec_db.std(axis=1)
        
        # Collect filename and features
        filename = os.path.basename(path)
        features = mel_mean.tolist() + mel_std.tolist()
        
        data.append([filename] + features)
    except Exception as e:
        print(f"Error processing {path}: {e}")

if data:
    # Create column names: mel1_mean, mel2_mean, ..., mel128_mean, mel1_std, ..., mel128_std
    n_mels = 128
    columns = (
        ["filename"] +
        [f"mel{i}_mean" for i in range(1, n_mels+1)] +
        [f"mel{i}_std" for i in range(1, n_mels+1)]
    )
    df = pd.DataFrame(data, columns=columns)
    df.to_csv("mel_spectrogram_features.csv", index=False)
    print("Mel spectrogram features saved to mel_spectrogram_features.csv")
else:
    print("No valid Mel spectrogram data found.")


Found 3241 .wav files


In [ ]:
import zipfile, io, soundfile as sf
import librosa, numpy as np, pandas as pd, os

zip_path = "ae_peru.zip"   # <-- your zip file
n_mels = 128
data = []

with zipfile.ZipFile(zip_path, "r") as z:
    wav_names = [n for n in z.namelist() if n.lower().endswith(".wav")]
    print(f"Found {len(wav_names)} .wav files in {zip_path}")

    for name in wav_names:
        try:
            # Read raw bytes and decode with soundfile
            with z.open(name, "r") as f:
                y, sr = sf.read(io.BytesIO(f.read()), dtype="float32", always_2d=False)
            if y.size == 0:
                print(f"Empty file: {name}")
                continue

            # Convert to mono if needed
            if y.ndim > 1:
                y = np.mean(y, axis=1)

            # Mel spectrogram
            mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels, fmax=sr//2)
            mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)

            # Summary stats over time
            mel_mean = mel_spec_db.mean(axis=1)
            mel_std  = mel_spec_db.std(axis=1)

            filename = os.path.basename(name)
            features = mel_mean.tolist() + mel_std.tolist()
            data.append([filename] + features)

        except Exception as e:
            print(f"Error processing {name}: {e}")

if data:
    columns = (
        ["filename"] +
        [f"mel{i}_mean" for i in range(1, n_mels+1)] +
        [f"mel{i}_std"  for i in range(1, n_mels+1)]
    )
    df = pd.DataFrame(data, columns=columns)
    out_csv = "mel_SDF_peru.csv"
    df.to_csv(out_csv, index=False)
    print(f"Mel spectrogram features saved to {out_csv}")
else:
    print("No valid Mel spectrogram data found.")


In [ ]:
import zipfile, tempfile, glob, os, csv, gc
import soundfile as sf
import librosa
import numpy as np
import pandas as pd

# ---------- SETTINGS ----------
ZIP_PATH = "ae_peru.zip"      # your zip
OUT_CSV  = "mel_SDF_peru.csv"
N_MELS   = 128
TARGET_SR = 22050             # downsample for stability/memory
MAX_SEC  = 600                # skip files longer than this (adjust)
# ------------------------------

def safe_sf_info(p):
    try:
        return sf.info(p)
    except Exception:
        return None

def feature_row(path):
    # Validate first
    info = safe_sf_info(path)
    if info is None:
        print(f"[skip] unreadable header: {path}")
        return None
    if info.frames > 0 and info.samplerate > 0:
        dur = info.frames / float(info.samplerate)
        if dur > MAX_SEC:
            print(f"[skip] too long ({dur:.1f}s): {path}")
            return None
    # Load with librosa (resample + mono)
    y, sr = librosa.load(path, sr=TARGET_SR, mono=True)
    if y.size == 0:
        print(f"[skip] empty audio: {path}")
        return None

    # Mel spectrogram
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS, fmax=sr//2)
    mel_db = librosa.power_to_db(mel, ref=np.max)

    mel_mean = mel_db.mean(axis=1)
    mel_std  = mel_db.std(axis=1)

    return [os.path.basename(path)] + mel_mean.tolist() + mel_std.tolist()

# Extract to temp and process
with tempfile.TemporaryDirectory() as tmpdir:
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(tmpdir)
    wavs = glob.glob(os.path.join(tmpdir, "**", "*.wav"), recursive=True)
    print(f"Found {len(wavs)} .wav files after extraction")

    # Prepare CSV writer (streaming)
    cols = (["filename"] +
            [f"mel{i}_mean" for i in range(1, N_MELS+1)] +
            [f"mel{i}_std"  for i in range(1, N_MELS+1)])
    with open(OUT_CSV, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(cols)

        for i, p in enumerate(wavs, 1):
            try:
                row = feature_row(p)
                if row is not None:
                    writer.writerow(row)
            except Exception as e:
                print(f"[skip] {p} -> {e}")
            if i % 50 == 0:
                gc.collect()  # keep memory tidy

print(f"Done. Wrote features to {OUT_CSV}")


Found 133 .wav files after extraction
